MLB_Fantasy_Dashboard Debugging


In [1]:
import json
import os
import numpy as np
import pandas as pd

from data.fetch_espn_data import mView_dict

from data.espn_mlb_utilities import get_league_info, get_roster_info, get_weekresults, get_category_stats, load_view_json, stat_melt_and_rename


2025-06-15 19:17:32.136 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-06-15 19:17:32.136 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
#Constants
LEAGUE_ID = 64175
YEAR = 2025
SCORING_PERIOD_ID = 10
ESPN_S2 = 'AEBAh2kfSRRp2FwlY9AZ0UvAnloth0cSRT9dN12S20HUcZL4XyxSvMbE%2BaTikFeweDzieUtsoKV710afOhydJ0pnV%2BZIrdfsCdQA%2FXdc9dKgEO4IlUgiUSICw0KJhiE12hwRDDelma%2B8Mc6s69bKQAkTk9sZlJ9jYuj8YBdupMYzRpJFxngDqjXUdZItScqzkp29X6jzZY7rb1UTSPdy7lTvigpuuDe4gVlEnrGigeQEyVRs62XjOv8XuT9OrksCqRfxaTp0pBWfMXUnvwxeAMgp'  # Cookie for authentication
SWID = '{3F15FCEB-EB45-4EE7-B1B1-92B76369484D}'
VIEWLIST = ('mTeam', 'mRoster', 'mBoxscore')

Load ESPN json's

In [3]:
def load_or_refresh_views(year: int, league_id: int, period_id: int, viewlist: tuple) -> dict:
    """Load ESPN data views from cache or API."""
    data = {}
    for view in viewlist:
        filename = f"{view}_{league_id}_{year}_{period_id}.json"
        filepath = os.path.join("data", "espn_json", filename)

        if os.path.exists(filepath):
            with open(filepath, "r") as f:
                print(f"Loaded cached {view}")#st.sidebar.info(f"Loaded cached {view}")
                data[view] = json.load(f)
        else:
            print(f"No cache for {view}, fetching...")#st.sidebar.warning(f"No cache for {view}, fetching...")
            data[view] = mView_dict(year, league_id, period_id, view)
            with open(filepath, "w") as f:
                json.dump(data[view], f)
            print(f"{view} saved.")#st.sidebar.success(f"{view} saved.")
    return data

In [4]:
data_views =load_or_refresh_views(YEAR, LEAGUE_ID, SCORING_PERIOD_ID, VIEWLIST)

MBOXSCORE = data_views['mBoxscore']
MTEAM = data_views['mTeam']
MROSTER = data_views['mRoster']

Loaded cached mTeam
Loaded cached mRoster
Loaded cached mBoxscore


On-Roster List, team name/player name

In [5]:
rosters_df = get_roster_info(MROSTER, MTEAM)

Get Weeky Results

** from data.espn_mlb_utilities import get_weekresults

In [6]:
from utils.id_maps import STATS_MAP
#create a reverse mapping
REVERSE_STATS_MAP = {v: k for k, v in STATS_MAP.items()}
# Main stat targets for 10-category league
TARGET_STATS = ['R', 'HR', 'RBI', 'SB', 'AVG', 'K', 'W', 'ERA', 'WHIP', 'SVHD']

# Additional components needed for ratios
STAT_COMPONENTS = {
    'AVG': ['H', 'AB'],
    'ERA': ['ER', 'OUTS'],
    'WHIP': ['P_BB', 'P_H', 'OUTS']
}
# Full list of all needed stats
ALL_STATS = set(TARGET_STATS)
for stat in TARGET_STATS:
    ALL_STATS.update(STAT_COMPONENTS.get(stat, []))

STAT_IDS = [REVERSE_STATS_MAP[s] for s in ALL_STATS]
STAT_DICT = {REVERSE_STATS_MAP[s]: s for s in ALL_STATS}

In [7]:
#def get_category_stats(league_id, year, scoring_period_id):
"""
Compute cleaned and corrected stat metrics for fantasy scoring.
"""

data = load_view_json("mBoxscore", LEAGUE_ID, YEAR, SCORING_PERIOD_ID)

if 'schedule' not in data or 'teams' not in data:
    raise ValueError("mBoxscore JSON is missing required keys: 'schedule' or 'teams'.")

df_schedule = pd.json_normalize(data['schedule'])[:-6]  # Drop playoff matchups
df_teams = pd.json_normalize(data['teams'])


2025-06-15 19:18:08.572 
  command:

    streamlit run c:\Users\frogg\Anaconda3\envs\FantasySports_py3_12_3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-06-15 19:18:08.574 No runtime found, using MemoryCacheStorageManager


In [ ]:
away.cumulativeScore.scoreByStat.45

def get_weekresults(df_schedule, df_teams):
"""Extracts week-by-week stat results for all teams, handling home/away symmetry."""

In [8]:
results = get_weekresults(df_schedule, df_teams)    

[41, 83, 0, 21, 45, 34, 1, 39, 2, 20, 53, 5, 48, 23, 37, 47]


In [13]:


home_keys = [f'home.cumulativeScore.scoreByStat.{id}' for id in STAT_IDS]
away_keys = [f'away.cumulativeScore.scoreByStat.{id}' for id in STAT_IDS]


In [15]:
f'away.cumulativeScore.scoreByStat.45.result'

'away.cumulativeScore.scoreByStat.45.result'

In [18]:
stat_melt_and_rename(df_schedule, ['away.teamId', 'matchupPeriodId'], [f'away.cumulativeScore.scoreByStat.45.score'], 'StatId', 'Score', 'away.teamId')

,teamId,matchupPeriodId,StatId,Score
0,NaN,1,45,NaN
1,NaN,1,45,NaN
2,NaN,1,45,NaN
3,NaN,1,45,NaN
4,NaN,1,45,NaN
...,...,...,...,...
121,4.0,21,45,NaN
122,8.0,21,45,NaN
123,5.0,21,45,NaN
124,1.0,21,45,NaN


In [19]:

home_results = stat_melt_and_rename(df_schedule, ['home.teamId', 'matchupPeriodId'],
                                    [f'{key}.result' for key in home_keys], 'StatId', 'Win/Loss', 'home.teamId')
away_results = stat_melt_and_rename(df_schedule, ['away.teamId', 'matchupPeriodId'],
                                    [f'{key}.result' for key in away_keys], 'StatId', 'Win/Loss', 'away.teamId')


In [20]:

home_scores = stat_melt_and_rename(df_schedule, ['home.teamId', 'matchupPeriodId'],
                                    [f'{key}.score' for key in home_keys], 'StatId', 'Score', 'home.teamId')
away_scores = stat_melt_and_rename(df_schedule, ['away.teamId', 'matchupPeriodId'],
                                    [f'{key}.score' for key in away_keys], 'StatId', 'Score', 'away.teamId')


In [24]:

winloss = pd.concat([home_results, away_results])
scores = pd.concat([home_scores, away_scores])

df = pd.merge(scores, winloss, on=['teamId', 'matchupPeriodId', 'StatId'], how='outer')


In [25]:

team_map = pd.Series(df_teams['name'].values, index=df_teams['id']).to_dict()
df['Team Names'] = df['teamId'].map(team_map)


In [26]:

df['Stat Name'] = df['StatId'].astype(int).map(STAT_DICT)
#df = df.dropna()


In [29]:

df['Score'] = pd.to_numeric(df['Score'], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)




In [ ]:

    results = recalculate_ratios(results)

    results = calculate_metrics(results)

    return results

In [9]:
weekly_results_df = get_category_stats(LEAGUE_ID, YEAR, SCORING_PERIOD_ID)

2025-06-15 19:18:43.201 No runtime found, using MemoryCacheStorageManager


[41, 83, 0, 21, 45, 34, 1, 39, 2, 20, 53, 5, 48, 23, 37, 47]


PAGE: PitcherList SP100

In [6]:
import pandas as pd

from data.espn_mlb_utilities import get_league_info,get_roster_info
from utils.rankings_utilities import get_tables, clean_players_names

In [7]:
# Define constants
LEAGUE_ID = 64175
YEAR = 2025
SCORING_PERIOD_ID = 8

In [8]:
#Get rosters, league info and combine (Add to front page??)
roster_df = pd.merge(get_roster_info(LEAGUE_ID, YEAR, SCORING_PERIOD_ID), get_league_info(LEAGUE_ID, YEAR, SCORING_PERIOD_ID), on='Team Number', how='left')


In [9]:
#Get Ranking

url = 'https://pitcherlist.com/top-100-starting-pitchers-for-2025-fantasy-baseball-week-8-5-19/'
table_number = 4                                

"""
Fetches and processes table data from the specified URL.

Args:
    url (str): The URL of the webpage containing the table.
    table_number (str): The identifier of the table on the webpage.
    roster_df (pd.DataFrame): DataFrame containing roster information.

Returns:
    pd.DataFrame: Processed DataFrame with pitcher list data.
"""
dfs = get_tables(url)
df = dfs[f'df_{table_number}'].copy()
df.rename(columns={'Pitcher': 'Player Names', 'Hitter': 'Player Names', 'Player': 'Player Names'}, inplace=True)
print (df)



    Rank         Player Names                                     Badges  \
0      1       Tarik SkubalT1               Aces Gonna AceQuality Starts   
1      2         Zack Wheeler               Aces Gonna AceQuality Starts   
2      3          Paul Skenes             Aces Gonna AceStrikeout Upside   
3      4         Jacob deGrom  Aces Gonna AceStrikeout UpsideInjury Risk   
4      5          Max FriedT2                   Aces Gonna AceWins Bonus   
..   ...                  ...                                        ...   
95    96     Justin Verlander      Streaming OptionWins BonusInjury Risk   
96    97         Ronel Blanco                 Streaming OptionWins Bonus   
97    98         Aaron Civale                 Streaming OptionWins Bonus   
98    99  Lance McCullers Jr.           Streaming OptionStrikeout Upside   
99   100        Bailey Falter             Streaming OptionQuality Starts   

   Change  
0       -  
1       -  
2       -  
3       -  
4      +1  
..    ...  
95 

In [10]:
#clean both df and Merge

roster_df_clean = clean_players_names(roster_df,'Player Names')
df_clean = clean_players_names(df,'Player Names')

In [11]:

ranking_roster_merged_df= pd.merge(df_clean, roster_df_clean[['Player Names', 'Team Names']], on='Player Names', how='left')
ranking_roster_merged_df['Team Names'] = ranking_roster_merged_df['Team Names'].fillna('Available')

# Unpacking the json(s) from espn fantasy  
 Views(List of EndPoints): 
 mDraftDetail  
 mLiveScoring  
 mMatchupScore  
 mPendingTransactions  
 mPositionalRatings  
 mSettings  
 mTeam  
              League overview, GM's and rankings, win/losses
 modular  
 mNav  
 kona_player_info  
 mTransactions2  
 mStatus  
 mPositionalRatingsStats  
 kona_game_state  
 proTeamSchedules_wl  
 players_wl  
 kona_league_communication  
 mRoster:  
            all playeres on all teams


mMatchup
mBoxscore
mSchedule
mScoreboard  



